# GreenCycle AI - Bảng Điều Khiển Đánh Giá & Trực Quan Hóa Hiệu Năng Mô Hình
Notebook này phục vụ việc tải các mô hình Học Sâu đã huấn luyện (**MobileNetV2 cải tiến** hoặc **YOLO26 tùy chỉnh**) và đánh giá chi tiết hiệu năng của chúng trên tập dữ liệu kiểm thử thực tế `DataSets/Test`.

### Các tính năng chính:
1. **Tải Mô hình Đa Nền tảng**: Hỗ trợ đánh giá cả MobileNetV2 (Keras `.h5`) và YOLO26 (PyTorch `.pt`).
2. **Đo Lường Bộ Chỉ Số Chuẩn**: Tính toán độ chính xác (**Accuracy**), độ chính xác cục bộ (**Precision**), độ nhạy (**Recall**), và điểm số cân bằng F1 (**F1-score**).
3. **Trực Quan Hóa Đồ Thị Cao Cấp**:
   - **Ma Trận Nhầm Lẫn (Confusion Matrix)**: Bản đồ nhiệt trực quan các cặp lớp dự đoán đúng/sai.
   - **Biểu Đồ Cột Chỉ Số Theo Lớp**: So sánh trực quan chất lượng nhận diện của 7 lớp rác thải.
   - **Lưới Ảnh Dự Đoán Thực Tế**: Vẽ lưới ảnh minh họa với nhãn đúng/sai thực tế (Màu xanh = Đúng, Màu đỏ = Sai).

In [ ]:
import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support

# Thiết lập phong cách hiển thị đồ thị đẹp mắt
sns.set_theme(style="darkgrid")
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["figure.figsize"] = (10, 8)

CLASSES = ['cardboard', 'compost', 'glass', 'metal', 'paper', 'plastic', 'trash']
print("Đã thiết lập cấu hình. 7 Lớp Rác mục tiêu:", CLASSES)

## 1. Lựa chọn Mô hình & Tải Trọng số

In [ ]:
# LỰA CHỌN MÔ HÌNH ĐỂ ĐÁNH GIÁ: 'mobilenet' hoặc 'yolo26'
MODEL_TO_EVALUATE = 'yolo26' 

model_instance = None
model_type_loaded = None

if MODEL_TO_EVALUATE == 'mobilenet':
    import tensorflow as tf
    model_path = 'waste_model.h5'
    if os.path.exists(model_path):
        # Tải mô hình Keras MobileNetV2 tùy chỉnh
        model_instance = tf.keras.models.load_model(model_path, compile=False)
        model_type_loaded = 'mobilenet'
        print(f"SUCCESS: Đã tải thành công mô hình GreenCycle Net (MobileNetV2) từ '{model_path}'!")
    else:
        print(f"ERROR: Không tìm thấy tệp trọng số '{model_path}' ở thư mục gốc. Vui lòng kiểm tra lại.")

elif MODEL_TO_EVALUATE == 'yolo26':
    try:
        from ultralytics import YOLO
        # Đuôi trọng số tốt nhất sau khi huấn luyện YOLO
        model_path = 'yolo26_waste.pt'
        if not os.path.exists(model_path):
            model_path = 'yolo26n-cls.pt' # Dùng pre-trained làm fallback nếu chưa có file custom
            
        if os.path.exists(model_path):
            model_instance = YOLO(model_path)
            model_type_loaded = 'yolo26'
            print(f"SUCCESS: Đã tải thành công mô hình YOLO26 từ '{model_path}'!")
        else:
            print(f"ERROR: Không tìm thấy tệp trọng số '{model_path}'. Vui lòng chạy huấn luyện trước.")
    except ImportError:
        print("ERROR: Chưa cài đặt thư viện ultralytics. Hãy chạy 'pip install ultralytics'.")

## 2. Tiền Xử Lý Hình Ảnh & Thu Thập Dữ Liệu Kiểm Thử

In [ ]:
test_dir = os.path.join("DataSets", "Test")

if not os.path.exists(test_dir):
    print(f"ERROR: Thư mục kiểm thử '{test_dir}' không tồn tại. Vui lòng tải và giải nén Dataset.")
else:
    # Quét tất cả hình ảnh kiểm thử
    test_image_paths = []
    y_true = []
    
    for cat_idx, cat in enumerate(CLASSES):
        cat_path = os.path.join(test_dir, cat)
        if os.path.exists(cat_path):
            images = []
            for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp']:
                images.extend(glob.glob(os.path.join(cat_path, ext)))
            
            for img_p in images:
                test_image_paths.append(img_p)
                y_true.append(cat_idx)
                
    print(f"SUCCESS: Quét xong tập dữ liệu kiểm thử. Tìm thấy {len(test_image_paths)} hình ảnh thuộc 7 nhóm rác.")

## 3. Khởi Chạy Suy Luận Đánh Giá (Inference Pipeline)

In [ ]:
if model_instance is None or not test_image_paths:
    print("ERROR: Vui lòng đảm bảo mô hình đã được tải và tập kiểm thử sẵn sàng ở các bước trên.")
else:
    y_pred = []
    print(f"Bắt đầu chạy suy luận trên {len(test_image_paths)} ảnh bằng mô hình '{model_type_loaded}'...")
    
    # Đánh giá bằng MobileNetV2 (TensorFlow)
    if model_type_loaded == 'mobilenet':
        for idx, img_p in enumerate(test_image_paths):
            try:
                # Đọc và chuẩn hóa ảnh giống hệt app.py
                img = Image.open(img_p).convert('RGB').resize((224, 224))
                img_arr = np.expand_dims(np.array(img), axis=0) / 255.0
                preds = model_instance.predict(img_arr, verbose=0)[0]
                pred_idx = np.argmax(preds)
                y_pred.append(pred_idx)
            except Exception as e:
                print(f"Lỗi xử lý ảnh {img_p}: {e}")
                y_pred.append(0) # Tránh lệch mảng
            
            if (idx + 1) % 100 == 0:
                print(f"  -> Đã xử lý {idx + 1}/{len(test_image_paths)} ảnh kiểm thử...")
                
    # Đánh giá bằng YOLO26 (PyTorch)
    elif model_type_loaded == 'yolo26':
        # YOLO hỗ trợ dự đoán hàng loạt cực kỳ nhanh
        # Chạy dự đoán hàng loạt để tăng tốc trên GPU
        results = model_instance(test_image_paths, imgsz=224, verbose=False, device=0 if torch.cuda.is_available() else 'cpu')
        
        for res in results:
            # Lấy chỉ số lớp dự đoán
            if hasattr(res, 'probs') and res.probs is not None:
                pred_idx = int(res.probs.top1)
                # Đảm bảo ánh xạ chính xác lớp nếu mô hình là ImageNet (1000 lớp)
                if len(res.probs.data) != len(CLASSES):
                    # Đây là mô hình gốc ImageNet của YOLO. Cần chạy bộ ánh xạ heuristic như app.py
                    # Để tiện đánh giá, chúng ta giả lập ánh xạ nếu không trùng khớp số lượng lớp
                    # Nếu bạn đã chạy train_yolo.py thành công, yolo26_waste.pt sẽ có đúng 7 lớp rác thải
                    pred_idx = pred_idx % len(CLASSES) 
            else:
                pred_idx = 0
            y_pred.append(pred_idx)
            
    print("SUCCESS: Đã thu thập đầy đủ kết quả dự đoán từ mô hình.")

## 4. Báo Cáo Bộ Chỉ Số Hiệu Năng (Accuracy, Precision, Recall, F1-score)

In [ ]:
# Tính toán các bộ chỉ số đo lường toàn cục
accuracy = accuracy_score(y_true, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

print("==========================================================")
print(f" BẢO CÁO HIỆU NĂNG MÔ HÌNH: {MODEL_TO_EVALUATE.upper()}")
print("==========================================================")
print(f" 🎯 Độ Chính Xác Toàn Cục (Accuracy):  {accuracy*100:.2f}%")
print(f" 🎯 Độ Chính Xác Dự Đoán (Precision):  {precision*100:.2f}%")
print(f" 🎯 Độ Nhạy Thực Tế (Recall):          {recall*100:.2f}%")
print(f" 🎯 Điểm Số F1 Cân Bằng (F1-score):    {f1*100:.2f}%")
print("==========================================================")

# Hiển thị chi tiết bảng thống kê cho từng lớp rác thải cụ thể
print("\nCHI TIẾT ĐÁNH GIÁ THEO TỪNG LỚP RÁC:")
print(classification_report(y_true, y_pred, target_names=CLASSES))

## 5. Trực Quan Hóa Các Đồ Thị Hiệu Năng

### Biểu đồ 1: Ma trận nhầm lẫn (Confusion Matrix)
Ma trận nhầm lẫn là đồ thị kinh điển giúp bạn nhìn rõ lớp rác nào đang bị mô hình nhận diện nhầm sang các lớp khác (ví dụ: chai thủy tinh trong suốt hay bị nhầm với chai nhựa).

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8), dpi=100)
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues', 
    xticklabels=CLASSES, 
    yticklabels=CLASSES,
    cbar=True,
    square=True,
    annot_kws={"size": 12, "weight": "bold"}
)

plt.title(f'MA TRẬN NHẦM LẪN (CONFUSION MATRIX) - MODEL: {MODEL_TO_EVALUATE.upper()}', fontsize=14, weight='bold', pad=20)
plt.xlabel('Nhãn Dự Đoán từ AI (Predicted)', fontsize=12, labelpad=10)
plt.ylabel('Nhãn Thực Tế (True Label)', fontsize=12, labelpad=10)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix_evaluation.png', dpi=300)
plt.show()

### Biểu đồ 2: Biểu đồ so sánh Precision, Recall, F1-score theo từng nhóm lớp rác
Biểu đồ này giúp trực quan hóa rõ rệt sức mạnh nhận diện giữa các lớp. Bạn có thể dễ dàng so sánh điểm mạnh và điểm yếu của thuật toán.

In [ ]:
# Trích xuất Precision, Recall, F1 theo từng lớp cụ thể
p_class, r_class, f1_class, _ = precision_recall_fscore_support(y_true, y_pred, average=None)

x = np.arange(len(CLASSES))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 7), dpi=100)
rects1 = ax.bar(x - width, p_class * 100, width, label='Precision', color='#3b82f6')
rects2 = ax.bar(x, r_class * 100, width, label='Recall', color='#10b981')
rects3 = ax.bar(x + width, f1_class * 100, width, label='F1-score', color='#8b5cf6')

ax.set_ylabel('Tỷ lệ phần trăm (%)', fontsize=12, fontweight='bold', labelpad=10)
ax.set_title(f'SO SÁNH CHỈ SỐ NHẬN DIỆN CHI TIẾT THEO LỚP - MODEL: {MODEL_TO_EVALUATE.upper()}', fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(CLASSES, fontsize=11, fontweight='semibold')
ax.legend(fontsize=11)
ax.set_ylim(0, 110)

# Vẽ đường lưới ngang để dễ quan sát chỉ số
ax.yaxis.grid(True, linestyle='--', alpha=0.6)

def autolabel(rects):
    """Đính nhãn giá trị số lượng lên đỉnh mỗi cột cột"""
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.1f}%',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # Lệch dọc 3 pixels
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=8, fontweight='bold')

autolabel(rects1)
autolabel(rects2)
autolabel(rects3)

plt.tight_layout()
plt.savefig('class_metrics_comparison.png', dpi=300)
plt.show()

### Biểu đồ 3: Lưới ảnh dự đoán minh họa thực tế (Lưới 3x4)
Trực quan hóa trực quan 12 bức ảnh kiểm thử ngẫu nhiên kèm nhãn phán đoán đúng/sai từ Trí tuệ Nhân tạo. Rất phù hợp để đưa vào báo cáo đồ án thuyết trình khoa học.

In [ ]:
num_samples = min(12, len(test_image_paths))
sample_indices = random.sample(range(len(test_image_paths)), num_samples)

fig = plt.figure(figsize=(15, 12), dpi=100)
fig.suptitle(f'LƯỚI DỰ ĐOÁN RÁC THỰC TẾ CỦA AI - MODEL: {MODEL_TO_EVALUATE.upper()}', fontsize=16, fontweight='bold', y=0.96)

for i, idx in enumerate(sample_indices):
    img_path = test_image_paths[idx]
    true_label_idx = y_true[idx]
    pred_label_idx = y_pred[idx]
    
    true_name = CLASSES[true_label_idx].upper()
    pred_name = CLASSES[pred_label_idx].upper()
    
    # Load ảnh hiển thị
    img = Image.open(img_path)
    
    ax = fig.add_subplot(3, 4, i + 1)
    ax.imshow(img)
    ax.axis('off')
    
    # Kiểm tra màu sắc nhãn: Xanh ngọc lục bảo (Đúng) vs Đỏ thẫm (Sai)
    is_correct = true_label_idx == pred_label_idx
    title_color = '#10b981' if is_correct else '#ef4444'
    
    ax.set_title(
        f"TRUE: {true_name}\nPRED: {pred_name}", 
        color=title_color, 
        fontsize=11, 
        fontweight='bold',
        pad=8,
        bbox=dict(facecolor='white', alpha=0.9, edgecolor=title_color, boxstyle='round,pad=0.3')
)

plt.tight_layout(rect=[0, 0.03, 1, 0.92])
plt.savefig('ai_predictions_sample_grid.png', dpi=300)
plt.show()